# AlphaTrain Pillar 2h: Elite Hybrid Training (Colab)

**Three fixes from Pillar 2g post-mortem:**
1. **Elite filter:** Only selfplay games scoring >= 1000 (220 games, 163K states, mean 1547)
2. **No cycling:** Selfplay drives epoch length, expert cycled to match. Each state ~1x per epoch.
3. **Policy from expert only:** Selfplay contributes ONLY value MSE, no policy CE.

Expert batches: `pol_CE + 0.001 * (ranking_loss + anchor_MSE)`
Self-play batches: `0.001 * MSE(value, TD_return)` — NO policy loss

**Parameters:** T=0.3 sharpening, LR=5e-5, 15 epochs, warm start from Pillar 2f

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_iter2_elite.pt.gz` — elite self-play data (52 MB)
3. `alphatrain_pairwise.pt` — expert data (already on Drive)
4. `pillar2f_best.pt` — base model (already on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model + expert data
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['pillar2f_best.pt', 'alphatrain_pairwise.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    # Always re-copy from Drive (force fresh)
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

# Decompress elite self-play data
gz_src = os.path.join(DRIVE, 'selfplay_iter2_elite.pt.gz')
pt_dst = '/content/alphatrain/data/selfplay_iter2_elite.pt'
# Always re-decompress from Drive (force fresh)
    print('Decompressing selfplay_iter2_elite.pt.gz...')
    t0 = time.time()
    !gzip -dc {gz_src} > {pt_dst}
    print(f'Done in {time.time()-t0:.0f}s: {os.path.getsize(pt_dst)/1e9:.1f} GB')
else:
    print(f'selfplay_iter2_elite.pt: {os.path.getsize(pt_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 30e9:
        print('WARNING: Need A100/H100 for hybrid training (two dataloaders).')

# Quick sanity checks
!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Verify both datasets
import torch

# Expert
ex = torch.load('/content/alphatrain/data/alphatrain_pairwise.pt', weights_only=True, map_location='cpu')
print(f"Expert: {ex['boards'].shape[0]:,} states, has_pairs={'good_boards' in ex}")
del ex

# Self-play (elite)
sp = torch.load('/content/alphatrain/data/selfplay_iter2_elite.pt', weights_only=False, map_location='cpu')
print(f"\nSelf-play (elite): {sp['observations'].shape[0]:,} states, {sp['n_games']} games")
print(f"  Min game score: {sp.get('min_game_score', 'N/A')}")
print(f"  Value range: {sp['value_targets'].min():.1f} - {sp['value_targets'].max():.1f}")
print(f"  Policy T: {sp.get('policy_temperature', 'N/A')}")
ent = -(sp['policy_targets'] * (sp['policy_targets'] + 1e-10).log()).sum(dim=-1).mean()
print(f"  Policy entropy: {ent:.2f}")
del sp

In [ ]:
# Pillar 2h: Elite Hybrid Training
#
# Expert (1.31M): pol_CE + 0.001 * (ranking + anchor) — cycled
# Self-play elite (163K, score>=1000): 0.001 * MSE value only — drives epoch
# Policy learns ONLY from expert. Value learns from both.
#
# Warm start from Pillar 2f (NOT 2g which regressed)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_hybrid \
    --expert-file alphatrain/data/alphatrain_pairwise.pt \
    --selfplay-file alphatrain/data/selfplay_iter2_elite.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 0.001 --rank-weight 1.0 --anchor-weight 0.001 \
    --resume alphatrain/data/pillar2f_best.pt --warm-start \
    --epochs 15 --batch-size 8192 --lr 5e-5 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2h_best.pt \
    --save-dir /content/checkpoints/pillar2h

In [ ]:
# Evaluate policy score (Pillar 2f baseline was 315/381)
# Should stay >= 310 (policy trained only on expert data)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2h/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2h/{f}'
    dst = f'{DRIVE}/pillar2h_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')